In [1]:
import pandas as pd
import numpy as np

In [2]:
# load csv
FILE_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_COMBINED.csv"

df = pd.read_csv(FILE_PATH)

print(f"Loaded {len(df)} rows")


Loaded 5436 rows


In [3]:
# clean practice number
df["PRACTICE_NUM"] = (
    df["PRACTICE_NUM"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan})
)

# simple address normalization
def normalize_address(x):
    if pd.isna(x):
        return ""
    return str(x).lower().replace(",", "").strip()

df["addr_norm"] = df["ADDRESS"].apply(normalize_address)

In [4]:
# sanity check
check = (
    df[df["PRACTICE_NUM"].notna()]
    .groupby("PRACTICE_NUM")
    .agg(
        n_rows=("PRACTICE_NUM", "count"),
        n_addresses=("ADDRESS", "nunique"),
        n_names=("PRACTICE_NAME", "nunique")
    )
    .reset_index()
)

conflicts = check[(check["n_addresses"] > 1) | (check["n_names"] > 1)]

print("\n--- SANITY CHECK ---")
print(f"Practice numbers with multiple addresses/names: {len(conflicts)}")

# show a few examples
print(conflicts.head())


--- SANITY CHECK ---
Practice numbers with multiple addresses/names: 1424
  PRACTICE_NUM  n_rows  n_addresses  n_names
0       100943       2            2        1
1      1013882       2            2        1
2      1015079       2            2        2
3      1015087       2            2        2
5      1015168       2            2        2


In [5]:
# grouping function that safely merges pharmacies with similar addresses and identical practice numbers
def dedupe_group(group):
    # if only one row → keep it
    if len(group) == 1:
        return group

    # unique normalized addresses
    unique_addrs = group["addr_norm"].unique()

    # if only 1 unique address → safe to merge
    if len(unique_addrs) == 1:
        return group.iloc[[0]]  # keep first

    # if addresses share key words (very simple check)
    # you can improve this later
    base = unique_addrs[0]
    similar = all(base[:15] in a for a in unique_addrs)

    if similar:
        return group.iloc[[0]]

    # otherwise DO NOT merge → keep all rows
    return group


In [6]:
with_id = df[df["PRACTICE_NUM"].notna()]
without_id = df[df["PRACTICE_NUM"].isna()]

deduped = (
    with_id
    .groupby("PRACTICE_NUM", group_keys=False)
    .apply(dedupe_group)
)

final_df = pd.concat([deduped, without_id], ignore_index=True)

print(f"Original: {len(df)}")
print(f"After: {len(final_df)}")

Original: 5436
After: 4862


In [7]:
# api call
import os
import time
import math
import requests


In [ ]:
# config
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
if not API_KEY:
    raise ValueError("GOOGLE_MAPS_API_KEY not found in environment variables.")

OUTPUT_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_pilot_results.csv"

# Keep this small at first
PILOT_N = 100
SLEEP_SECONDS = 0.2

In [15]:
# helper functions

def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def build_query(row):
    """
    Build a conservative text query from the fields you already have.
    """
    practice_name = clean_text(row.get("PRACTICE_NAME"))
    address = clean_text(row.get("ADDRESS"))
    city = clean_text(row.get("CITY"))
    province = clean_text(row.get("PROVINCE"))

    parts = []

    if practice_name:
        parts.append(practice_name)
    if address:
        parts.append(address)
    if city:
        parts.append(city)
    if province:
        parts.append(province)

    parts.append("South Africa")

    return ", ".join(parts)


def search_place_text(query, api_key):
    url = "https://maps.googleapis.com/maps/api/place/textsearch/json"

    params = {
        "query": query,
        "type": "pharmacy",  # helps filter results
        "key": api_key
    }

    r = requests.get(url, params=params, timeout=30)
    http_status = r.status_code

    try:
        data = r.json()
    except Exception:
        return {
            "http_status": http_status,
            "api_status": "BAD_JSON",
            "place_id": None,
            "matched_name": None,
            "matched_address": None,
            "lat": None,
            "lng": None,
            "types": None,
            "raw_error": r.text[:500]
        }

    status = data.get("status")

    if status != "OK":
        return {
            "http_status": http_status,
            "api_status": status,
            "place_id": None,
            "matched_name": None,
            "matched_address": None,
            "lat": None,
            "lng": None,
            "types": None,
            "raw_error": str(data)[:500]
        }

    result = data["results"][0]

    return {
        "http_status": http_status,
        "api_status": status,
        "place_id": result.get("place_id"),
        "matched_name": result.get("name"),
        "matched_address": result.get("formatted_address"),
        "lat": result.get("geometry", {}).get("location", {}).get("lat"),
        "lng": result.get("geometry", {}).get("location", {}).get("lng"),
        "types": ", ".join(result.get("types", [])),
        "raw_error": None
    }


def needs_review(row):
    if row["api_status"] != "OK":
        return True
    if pd.isna(row["place_id"]):
        return True
    if pd.isna(row["lat"]) or pd.isna(row["lng"]):
        return True
    return False


In [16]:
# query strings
places_df = final_df.copy()
places_df["query_string"] = places_df.apply(build_query, axis=1)

# Only keep rows that actually have a usable query
places_df = places_df[
    places_df["query_string"].notna() &
    (places_df["query_string"].str.len() > 0)
].copy()

# Start with a small pilot
places_df = places_df.head(PILOT_N).copy()

print(f"Running Places API on {len(places_df)} rows")

Running Places API on 100 rows


In [17]:
# api call
results = []

for idx, row in places_df.iterrows():
    query = row["query_string"]
    result = search_place_text(query, API_KEY)

    results.append({
        "raw_row_id": row.get("raw_row_id"),
        "PHARMACY_ID": row.get("PHARMACY_ID"),
        "PRACTICE_NUM": row.get("PRACTICE_NUM"),
        "PRACTICE_NAME": row.get("PRACTICE_NAME"),
        "ADDRESS": row.get("ADDRESS"),
        "CITY": row.get("CITY"),
        "PROVINCE": row.get("PROVINCE"),
        "PHONE": row.get("PHONE"),
        "query_string": query,
        **result
    })

    if len(results) % 10 == 0:
        print(f"Processed {len(results)} / {len(places_df)}")

    time.sleep(SLEEP_SECONDS)


Processed 10 / 100
Processed 20 / 100
Processed 30 / 100
Processed 40 / 100
Processed 50 / 100
Processed 60 / 100
Processed 70 / 100
Processed 80 / 100
Processed 90 / 100
Processed 100 / 100


In [18]:
results_df = pd.DataFrame(results)
results_df["needs_review"] = results_df.apply(needs_review, axis=1)

results_df.to_csv(OUTPUT_PATH, index=False)

print("\nDone.")
print(f"Saved results to:\n{OUTPUT_PATH}")

print("\nAPI status counts:")
print(results_df["api_status"].value_counts(dropna=False))

print("\nNeeds review counts:")
print(results_df["needs_review"].value_counts(dropna=False))


Done.
Saved results to:
C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_pilot_results.csv

API status counts:
OK              98
ZERO_RESULTS     2
Name: api_status, dtype: int64

Needs review counts:
False    98
True      2
Name: needs_review, dtype: int64


In [19]:
FINAL_DF_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_FINAL_DEDUPED.csv"

final_df.to_csv(FINAL_DF_PATH, index=False)